# Passerine Body Plan Evolution, Part 2: Lineage-Rate Summaries

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/jakeberv/bifrost/blob/main/vignettes/colab/avian-skeleton-part-2.ipynb)

Converted from [`vignettes/avian-skeleton-part-2.Rmd`](https://github.com/jakeberv/bifrost/blob/main/vignettes/avian-skeleton-part-2.Rmd).

> **Recommended Colab runtime:** For compute-intensive examples, choose **Runtime > Change runtime type > v5e-1 TPU** when available. This notebook uses the runtime's host CPUs, not the TPU itself. Otherwise, choose the available runtime with the most CPUs.

In [ ]:
message("Detected logical CPUs: ", parallel::detectCores(logical = TRUE))
if (!dir.exists("/content/bifrost")) {
  system("git clone --depth 1 https://github.com/jakeberv/bifrost.git /content/bifrost")
}
colab_packages <- c("remotes", "knitr", "ggplot2", "patchwork")
missing_packages <- colab_packages[
  !vapply(colab_packages, requireNamespace, logical(1), quietly = TRUE)
]
if (length(missing_packages)) {
  install.packages(missing_packages, repos = "https://cloud.r-project.org")
}
remotes::install_local("/content/bifrost", dependencies = NA, upgrade = "never")
setwd("/content/bifrost/vignettes")


In [ ]:
library(bifrost)

knitr::opts_chunk$set(
  collapse = TRUE,
  comment = "#>"
)

vignette_figure_caption <- function(html, latex) {
  pandoc_to <- knitr::opts_knit$get("rmarkdown.pandoc.to")
  if (is.character(pandoc_to) && length(pandoc_to) == 1L && grepl("^html", pandoc_to)) html else latex
}

is_part2_html_output <- function() {
  pandoc_to <- knitr::opts_knit$get("rmarkdown.pandoc.to")
  isTRUE(knitr::is_html_output()) &&
    is.character(pandoc_to) &&
    length(pandoc_to) == 1 &&
    grepl("^html", pandoc_to)
}

# Keep the interactive and static Figure 1 toy examples synchronized.
lineage_decay_widget <- list(
  defaults = list(
    decay_base = 2,
    half_life = 5,
    max_age = 25,
    normalize_weights = TRUE
  ),
  tree_height = 20,
  example_ages = c(0, 5, 10, 15, 20),
  state_rates = c(
    slow = 0.0011,
    intermediate = 0.0024,
    fast = 0.0034
  ),
  path_nodes = data.frame(
    element = 1:4,
    age = c(20, 8, 4, 2),
    younger_age = c(8, 4, 2, 0),
    y = c(205, 205, 245, 273),
    state = c("slow", "intermediate", "fast", "fast"),
    stringsAsFactors = FALSE
  )
)

lineage_decay_widget_json <- function(settings) {
  path_json <- vapply(seq_len(nrow(settings$path_nodes)), function(i) {
    node <- settings$path_nodes[i, ]
    sprintf(
      '{"element":%d,"age":%g,"youngerAge":%g,"y":%g,"state":"%s"}',
      node$element,
      node$age,
      node$younger_age,
      node$y,
      node$state
    )
  }, character(1))
  rate_json <- paste(
    sprintf(
      '"%s":%g',
      names(settings$state_rates),
      unname(settings$state_rates)
    ),
    collapse = ","
  )
  sprintf(
    paste0(
      '{"defaults":{"decayBase":%g,"halfLife":%g,"maxAge":%g,',
      '"normalizeWeights":%s},"treeHeight":%g,"exampleAges":[%s],',
      '"rates":{%s},"pathNodes":[%s]}'
    ),
    settings$defaults$decay_base,
    settings$defaults$half_life,
    settings$defaults$max_age,
    if (isTRUE(settings$defaults$normalize_weights)) "true" else "false",
    settings$tree_height,
    paste(settings$example_ages, collapse = ","),
    rate_json,
    paste(path_json, collapse = ",")
  )
}


_This pkgdown-only HTML chunk was omitted from the Colab notebook._


## Introduction

Part 2 asks how a fitted multi-regime Brownian motion model (BMM) can be turned into a species-level summary of evolutionary tempo. The focal `bifrost` search estimates scalar transformations of a multivariate phenotypic variance-covariance matrix on a mapped phylogeny; those fitted rates live on regimes and ancestral histories, not directly in a tidy table of extant species. `lineage_rates()` provides that bridge by collapsing each extant species' inherited regime history into a present-biased lineage-rate statistic.

Berv et al. (2026) used this statistic as a way to ask whether recent evolutionary tempo inferred from deep-time rate shifts relates to contemporary species-level and assemblage-level predictors. This vignette stays at the reusable package step: it shows how the final mapped tree and fitted regime rates become one present-weighted summary per tip, while keeping the terminal fitted rate available as a separate comparison. The compact object is enough for this calculation; the full archive remains the source for proposal history, sensitivity searches, and candidate model fits.

## Computing lineage-rate summaries

The purpose of `lineage_rates()` is to summarize the rate history inherited by each extant lineage when the terminal regime alone is too narrow a description. Two species can share the same terminal fitted rate after inheriting different ancestral regimes, and two lineages with similar recent rates can carry different amounts of deeper history. The statistic therefore treats the mapped BMM as a piecewise-constant history of multivariate evolutionary tempo and computes a temporally filtered average along each root-to-tip path.

In the avian body-plan example, the state value being averaged is the scalar BMM rate assigned to each mapped regime, interpreted by Berv et al. (2026) as the mean of the diagonal variances in that regime's evolutionary variance-covariance matrix. The resulting `log_lineage_rate` is not an instantaneous tip-rate estimate or a hard recent-time-window average. It is a tunable, present-biased summary of inherited regime history, reported by Berv et al. (2026) under `decay_base = 2`, `half_life = 5`, and normalized weights.

The key object is an age-decay kernel. Following the `lineage_rates()` documentation, the normalized weighted lineage-rate statistic is

\[
\bar{r}_{\mathrm{WLR}} =
\frac{1}{Z} \sum_{i=1}^{L} b^{-a_i/T} r_i
\]

where

\[
Z = \sum_{j=1}^{L} b^{-a_j/T}.
\]

Here \(r_i\) is the fitted value for ancestral node or path element \(i\), \(a_i\) is that element's age before the selected reference point, \(b\) is `decay_base`, \(T\) is the `half_life` argument used as a decay time scale, and \(L\) is the number of included path elements. For the top-level statistic used here, \(i\) indexes ancestral nodes along the lineage, including the root and excluding the terminal tip state. When `half_life = NULL`, the kernel is \(b^{-a_i}\) instead of \(b^{-a_i/T}\); when `normalize_weights = FALSE`, the \(1/Z\) normalization is skipped.

The name `half_life` is literal only when `decay_base = 2`: a path element \(T\) Myr older then receives \(2^{-1} = 1/2\) the raw weight of a present-day element. More generally, an element \(T\) Myr older receives \(1/b\) the raw weight, so the slider below labels \(T\) as the decay scale while noting the manuscript half-life setting. Figure 1 shows how the settings interact, then traces those same settings through a miniature mapped lineage before the lineage-rate table is computed.

### How decay base and decay scale shape lineage-rate memory


_This pkgdown-only HTML chunk was omitted from the Colab notebook._


In [ ]:
decay_kernel_ages <- seq(0, lineage_decay_widget$defaults$max_age, length.out = 300)

decay_kernel_curve <- function(label, decay_base, half_life, ages = decay_kernel_ages) {
  weight <- if (decay_base == 1) {
    rep(1, length(ages))
  } else if (is.null(half_life) || is.na(half_life)) {
    decay_base^(-ages)
  } else {
    decay_base^(-ages / half_life)
  }
  data.frame(
    age = ages,
    weight = weight,
    label = label,
    stringsAsFactors = FALSE
  )
}

decay_kernel_theme <- ggplot2::theme_classic(base_size = 8.5) +
  ggplot2::theme(
    legend.position = "top",
    legend.title = ggplot2::element_blank(),
    legend.margin = ggplot2::margin(0, 0, 0, 0),
    plot.title = ggplot2::element_text(face = "bold", color = "#263238", size = 9),
    plot.subtitle = ggplot2::element_text(color = "#5b646b", size = 8),
    plot.margin = ggplot2::margin(4, 4, 4, 4),
    axis.title = ggplot2::element_text(color = "#263238"),
    axis.text = ggplot2::element_text(color = "#263238")
  )

half_life_levels <- c(
  "0.5",
  "2",
  "5_default",
  "10"
)
half_life_data <- do.call(rbind, list(
  decay_kernel_curve("0.5", 2, 0.5),
  decay_kernel_curve("2", 2, 2),
  decay_kernel_curve("5_default", 2, 5),
  decay_kernel_curve("10", 2, 10)
))
half_life_data$label <- factor(half_life_data$label, levels = half_life_levels)

example_age_values <- lineage_decay_widget$example_ages
example_weight_specs <- data.frame(
  label = c("Equal weights", "5_default", "0.5"),
  half_life = c(Inf, 5, 0.5),
  stringsAsFactors = FALSE
)
example_weight_data <- do.call(rbind, lapply(seq_len(nrow(example_weight_specs)), function(i) {
  spec <- example_weight_specs[i, ]
  raw <- if (is.infinite(spec$half_life)) {
    rep(1, length(example_age_values))
  } else {
    2^(-example_age_values / spec$half_life)
  }
  data.frame(
    age = factor(paste0(example_age_values, " Myr"), levels = paste0(example_age_values, " Myr")),
    share = raw / sum(raw),
    label = spec$label,
    stringsAsFactors = FALSE
  )
}))
example_weight_data$label <- factor(example_weight_data$label, levels = example_weight_specs$label)

half_life_colors <- c("#8f3d46", "#d98c00", "#2f7f5f", "#2f6f9f")
example_weight_colors <- c("#5f6368", "#2f7f5f", "#8f3d46")

decay_panel_a <- ggplot2::ggplot(half_life_data, ggplot2::aes(x = age, y = weight, color = label)) +
  ggplot2::geom_hline(yintercept = 0.5, color = "#c7cdd4", linewidth = 0.30, linetype = "dashed") +
  ggplot2::geom_line(linewidth = 0.85) +
  ggplot2::scale_color_manual(
    values = half_life_colors,
    labels = c(
      expression(t[1/2] == 0.5),
      expression(t[1/2] == 2),
      expression(t[1/2] == 5~"(default)"),
      expression(t[1/2] == 10)
    )
  ) +
  ggplot2::scale_y_continuous(limits = c(0, 1.02), expand = ggplot2::expansion(mult = c(0, 0.02))) +
  ggplot2::labs(
    title = "A. Raw age-decay kernel",
    subtitle = "decay_base fixed at b = 2",
    x = "Age before present (Myr)",
    y = "Raw weight"
  ) +
  decay_kernel_theme

decay_panel_b <- ggplot2::ggplot(example_weight_data, ggplot2::aes(x = age, y = share, fill = label)) +
  ggplot2::geom_col(position = ggplot2::position_dodge(width = 0.76), width = 0.68) +
  ggplot2::scale_fill_manual(
    values = example_weight_colors,
    labels = c(
      "Equal weights",
      expression(t[1/2] == 5~"(default)"),
      expression(t[1/2] == 0.5)
    )
  ) +
  ggplot2::scale_y_continuous(limits = c(0, 1), expand = ggplot2::expansion(mult = c(0, 0.02))) +
  ggplot2::labs(
    title = "B. Normalized shares for one lineage",
    subtitle = "example path elements at 0, 5, 10, 15, and 20 Myr",
    x = "Element age before present",
    y = "Normalized share of total weight"
  ) +
  decay_kernel_theme +
  ggplot2::theme(legend.position = "right")

static_kernel_row <- patchwork::wrap_plots(
  decay_panel_a,
  decay_panel_b,
  ncol = 2,
  widths = c(1.05, 1)
)

schematic_base <- lineage_decay_widget$defaults$decay_base
schematic_half_life <- lineage_decay_widget$defaults$half_life

simmap_tree <- ape::read.tree(
  text = "((alpha:12,beta:12):8,(gamma:8,(delta:4,(epsilon:2,zeta:2):2):4):12);"
)

state_levels <- c("slow", "intermediate", "fast")
state_colors <- c(
  slow = "#4c78a8",
  intermediate = "#f58518",
  fast = "#b279a2",
  background = "#c8d0d6"
)
summary_color <- "#4c78a8"
state_rates <- lineage_decay_widget$state_rates

root_node <- length(simmap_tree$tip.label) + 1
focal_tip <- which(simmap_tree$tip.label == "epsilon")
focal_path <- ape::nodepath(simmap_tree, from = root_node, to = focal_tip)
focal_edges <- integer(length(focal_path) - 1)
for (i in seq_along(focal_edges)) {
  focal_edges[i] <- which(
    simmap_tree$edge[, 1] == focal_path[i] &
      simmap_tree$edge[, 2] == focal_path[i + 1]
  )
}

simmap_maps <- vector("list", length(simmap_tree$edge.length))
for (i in seq_along(simmap_maps)) {
  simmap_maps[[i]] <- stats::setNames(simmap_tree$edge.length[i], "slow")
}
focal_edge_states <- lineage_decay_widget$path_nodes$state
for (i in seq_along(focal_edges)) {
  simmap_maps[[focal_edges[i]]] <- stats::setNames(
    simmap_tree$edge.length[focal_edges[i]],
    focal_edge_states[i]
  )
}

simmap_tree$maps <- simmap_maps
simmap_tree$mapped.edge <- matrix(
  0,
  nrow = length(simmap_tree$edge.length),
  ncol = length(state_levels),
  dimnames = list(NULL, state_levels)
)
for (i in seq_along(simmap_tree$maps)) {
  for (state in names(simmap_tree$maps[[i]])) {
    simmap_tree$mapped.edge[i, state] <- simmap_tree$mapped.edge[i, state] +
      simmap_tree$maps[[i]][[state]]
  }
}
class(simmap_tree) <- c("simmap", class(simmap_tree))
stopifnot(inherits(simmap_tree, "simmap"))

node_x <- ape::node.depth.edgelength(simmap_tree)
tree_height <- max(node_x[seq_along(simmap_tree$tip.label)])
stopifnot(isTRUE(all.equal(tree_height, lineage_decay_widget$tree_height)))
node_y <- numeric(length(node_x))
node_y[seq_along(simmap_tree$tip.label)] <- rev(seq_along(simmap_tree$tip.label))
for (node in rev(seq.int(root_node, length(node_x)))) {
  children <- simmap_tree$edge[simmap_tree$edge[, 1] == node, 2]
  if (length(children) > 0) {
    node_y[node] <- mean(node_y[children])
  }
}

vertical_segments <- data.frame()
for (node in seq.int(root_node, length(node_x))) {
  children <- simmap_tree$edge[simmap_tree$edge[, 1] == node, 2]
  if (length(children) > 0) {
    vertical_segments <- rbind(
      vertical_segments,
      data.frame(
        x = node_x[node],
        xend = node_x[node],
        y = min(node_y[children]),
        yend = max(node_y[children])
      )
    )
  }
}

horizontal_segments <- data.frame()
for (edge_index in seq_len(nrow(simmap_tree$edge))) {
  parent <- simmap_tree$edge[edge_index, 1]
  child <- simmap_tree$edge[edge_index, 2]
  is_focal <- edge_index %in% focal_edges

  if (is_focal) {
    segment_start <- node_x[parent]
    for (map_index in seq_along(simmap_tree$maps[[edge_index]])) {
      segment_length <- simmap_tree$maps[[edge_index]][[map_index]]
      segment_end <- segment_start + segment_length
      horizontal_segments <- rbind(
        horizontal_segments,
        data.frame(
          x = segment_start,
          xend = segment_end,
          y = node_y[child],
          yend = node_y[child],
          state = names(simmap_tree$maps[[edge_index]])[map_index],
          focal = TRUE
        )
      )
      segment_start <- segment_end
    }
  } else {
    horizontal_segments <- rbind(
      horizontal_segments,
      data.frame(
        x = node_x[parent],
        xend = node_x[child],
        y = node_y[child],
        yend = node_y[child],
        state = "background",
        focal = FALSE
      )
    )
  }
}

included_nodes <- focal_path[-length(focal_path)]
path_node_data <- data.frame(
  node = included_nodes,
  edge = focal_edges,
  element = seq_along(included_nodes),
  x = node_x[included_nodes],
  y = node_y[included_nodes],
  age = tree_height - node_x[included_nodes],
  younger_age = tree_height - node_x[focal_path[-1]],
  stringsAsFactors = FALSE
)
path_node_data$state <- vapply(
  path_node_data$edge,
  function(edge_index) names(simmap_tree$maps[[edge_index]])[1],
  character(1)
)
stopifnot(
  identical(path_node_data$element, lineage_decay_widget$path_nodes$element),
  isTRUE(all.equal(path_node_data$age, lineage_decay_widget$path_nodes$age)),
  isTRUE(all.equal(path_node_data$younger_age, lineage_decay_widget$path_nodes$younger_age)),
  identical(path_node_data$state, lineage_decay_widget$path_nodes$state)
)
path_node_data$rate <- unname(state_rates[path_node_data$state])
path_node_data$raw_weight <- schematic_base^(-path_node_data$age / schematic_half_life)
path_node_data$weight <- path_node_data$raw_weight / sum(path_node_data$raw_weight)
path_node_data$contribution <- path_node_data$weight * path_node_data$rate
path_node_data$age_label <- paste0(path_node_data$age, " Myr")
path_node_data$age_factor <- factor(path_node_data$age_label, levels = path_node_data$age_label)
path_node_data$label_x <- path_node_data$x
path_node_data$label_y <- path_node_data$y + 0.48
path_node_data$label_hjust <- 0.5
last_path_element <- path_node_data$element == max(path_node_data$element)
path_node_data$label_x[last_path_element] <- path_node_data$x[last_path_element] - 0.65
path_node_data$label_y[last_path_element] <- path_node_data$y[last_path_element] + 0.62
path_node_data$label_hjust[last_path_element] <- 1
path_node_data$weight_label <- sprintf(
  "w[%s] == %.2f",
  path_node_data$element,
  path_node_data$weight
)
path_node_data$rate_label <- sprintf(
  "r[%s] == %.4f",
  path_node_data$element,
  path_node_data$rate
)
path_node_data$interval_min <- pmin(path_node_data$age, path_node_data$younger_age)
path_node_data$interval_max <- pmax(path_node_data$age, path_node_data$younger_age)
path_node_data$interval_mid <- (path_node_data$age + path_node_data$younger_age) / 2
path_node_data$detail_x <- path_node_data$interval_mid
path_node_data$detail_x[path_node_data$element == 3] <- path_node_data$detail_x[path_node_data$element == 3] + 0.55
path_node_data$detail_x[path_node_data$element == 4] <- path_node_data$detail_x[path_node_data$element == 4] - 0.35

tip_data <- data.frame(
  label = simmap_tree$tip.label,
  x = node_x[seq_along(simmap_tree$tip.label)],
  y = node_y[seq_along(simmap_tree$tip.label)]
)
tip_data$focal <- tip_data$label == "epsilon"

vertical_segments$x <- tree_height - vertical_segments$x
vertical_segments$xend <- tree_height - vertical_segments$xend
horizontal_segments$x <- tree_height - horizontal_segments$x
horizontal_segments$xend <- tree_height - horizontal_segments$xend
path_node_data$x <- tree_height - path_node_data$x
path_node_data$label_x <- tree_height - path_node_data$label_x
tip_data$x <- tree_height - tip_data$x

schematic_tree_panel <- ggplot2::ggplot() +
  ggplot2::geom_segment(
    data = vertical_segments,
    ggplot2::aes(
      x = x,
      y = y,
      xend = xend,
      yend = yend
    ),
    color = "#d5dbe0",
    linewidth = 0.45,
    lineend = "round"
  ) +
  ggplot2::geom_segment(
    data = horizontal_segments,
    ggplot2::aes(
      x = x,
      y = y,
      xend = xend,
      yend = yend,
      color = state,
      linewidth = focal
    ),
    lineend = "round"
  ) +
  ggplot2::scale_color_manual(
    values = state_colors,
    breaks = state_levels,
    labels = c("slow", "intermediate", "fast"),
    name = "Mapped regime"
  ) +
  ggplot2::scale_linewidth_manual(
    values = c("TRUE" = 1.35, "FALSE" = 0.55),
    guide = "none"
  ) +
  ggplot2::geom_point(
    data = path_node_data,
    ggplot2::aes(x = x, y = y, fill = state),
    shape = 21,
    color = "white",
    stroke = 0.35,
    size = 2.4
  ) +
  ggplot2::scale_fill_manual(values = state_colors[state_levels], guide = "none") +
  ggplot2::geom_point(
    data = tip_data[tip_data$focal, ],
    ggplot2::aes(x = x, y = y),
    shape = 21,
    fill = "white",
    color = "#263238",
    size = 2.4,
    stroke = 0.55
  ) +
  ggplot2::geom_text(
    data = tip_data[!tip_data$focal, ],
    ggplot2::aes(x = x - 0.5, y = y, label = label),
    color = "#8a949b",
    hjust = 0,
    size = 2.35
  ) +
  ggplot2::geom_text(
    data = tip_data[tip_data$focal, ],
    ggplot2::aes(x = x - 0.5, y = y, label = label),
    color = "#263238",
    hjust = 0,
    fontface = "bold",
    size = 2.45
  ) +
  ggplot2::annotate(
    "text",
    x = 10,
    y = 6.55,
    label = "trace one extant lineage through mapped regimes",
    color = "#263238",
    fontface = "bold",
    size = 3.05
  ) +
  ggplot2::scale_x_reverse(
    limits = c(tree_height + 0.7, -3.2),
    breaks = seq(tree_height, 0, by = -5),
    expand = c(0, 0)
  ) +
  ggplot2::coord_cartesian(ylim = c(0.45, 6.85), clip = "off") +
  ggplot2::labs(
    title = "C. Trace a lineage through a simmap tree",
    x = "Age before present (Myr)",
    y = NULL
  ) +
  ggplot2::theme_classic(base_size = 8.5) +
  ggplot2::theme(
    plot.title = ggplot2::element_text(face = "bold", color = "#263238", size = 9),
    plot.margin = ggplot2::margin(4, 8, 4, 4),
    axis.text.y = ggplot2::element_blank(),
    axis.ticks.y = ggplot2::element_blank(),
    axis.line.y = ggplot2::element_blank(),
    axis.title.x = ggplot2::element_text(color = "#263238"),
    axis.text.x = ggplot2::element_text(color = "#263238"),
    legend.position = "top",
    legend.justification = "left",
    legend.title = ggplot2::element_text(color = "#263238", size = 7.5),
    legend.text = ggplot2::element_text(color = "#263238", size = 7.5),
    legend.key.width = grid::unit(0.55, "lines"),
    legend.margin = ggplot2::margin(0, 0, 0, 0)
  )

schematic_weight_panel <- ggplot2::ggplot(path_node_data) +
  ggplot2::geom_rect(
    ggplot2::aes(
      xmin = interval_min,
      xmax = interval_max,
      ymin = 0,
      ymax = weight,
      fill = state
    ),
    alpha = 0.92
  ) +
  ggplot2::scale_fill_manual(values = state_colors[state_levels], guide = "none") +
  ggplot2::geom_text(
    ggplot2::aes(x = detail_x, y = -0.070, label = weight_label),
    color = "#263238",
    size = 1.85,
    parse = TRUE
  ) +
  ggplot2::geom_text(
    ggplot2::aes(x = detail_x, y = -0.125, label = rate_label),
    color = "#263238",
    size = 1.75,
    parse = TRUE
  ) +
  ggplot2::annotate(
    "text",
    x = tree_height - 0.6,
    y = 0.94,
    label = sprintf("rWLR = sum(w_i * r_i) = %.4f", sum(path_node_data$contribution)),
    hjust = 0,
    color = summary_color,
    fontface = "bold",
    size = 4.5
  ) +
  ggplot2::annotate(
    "text",
    x = tree_height / 2,
    y = -0.025,
    label = "Age before present (Myr)",
    color = "#263238",
    fontface = "bold",
    size = 2.4
  ) +
  ggplot2::scale_x_reverse(
    limits = c(tree_height, 0),
    breaks = seq(tree_height, 0, by = -5),
    expand = c(0, 0)
  ) +
  ggplot2::scale_y_continuous(
    breaks = seq(0, 1, by = 0.25),
    expand = ggplot2::expansion(mult = c(0, 0.02))
  ) +
  ggplot2::coord_cartesian(ylim = c(-0.16, 1), clip = "off") +
  ggplot2::labs(
    title = "D. Weight calculation",
    subtitle = sprintf("b = %s, T = %s Myr; T is a half-life only when b = 2", schematic_base, schematic_half_life),
    x = NULL,
    y = expression("Normalized weight " * w[i])
  ) +
  ggplot2::theme_classic(base_size = 8.5) +
  ggplot2::theme(
    plot.title = ggplot2::element_text(face = "bold", color = "#263238", size = 9),
    plot.subtitle = ggplot2::element_text(color = "#5b646b", size = 8),
    axis.title = ggplot2::element_text(color = "#263238"),
    axis.text = ggplot2::element_text(color = "#263238"),
    plot.margin = ggplot2::margin(4, 4, 4, 8)
  )

static_lineage_row <- patchwork::wrap_plots(
  schematic_tree_panel,
  schematic_weight_panel,
  ncol = 2,
  widths = c(1.25, 1)
)

patchwork::wrap_plots(
  static_kernel_row,
  static_lineage_row,
  ncol = 1,
  heights = c(1, 1.12)
)


### Loading the focal search


In [ ]:
# Load the same compact focal result inspected in Part 1.
search_path <- system.file(
  "extdata",
  "avian-skeleton",
  "passerine_bodyplan_search_compact.RDS",
  package = "bifrost"
)
stopifnot(nzchar(search_path))

bodyplan_search <- readRDS(search_path)

# Preserve the S3 class if an older serialized copy lacks it.
if (!inherits(bodyplan_search, "bifrost_search")) {
  class(bodyplan_search) <- c("bifrost_search", class(bodyplan_search))
}

# Confirm that the object is a fitted BMM search with inferred shifts.
data.frame(
  inherits_bifrost_search = inherits(bodyplan_search, "bifrost_search"),
  model = bodyplan_search$model_no_uncertainty$model,
  fitted_regimes = length(bodyplan_search$model_no_uncertainty$param),
  accepted_shifts = length(bodyplan_search$shift_nodes_no_uncertainty)
)


This check confirms that the compact object is a multi-regime BMM search, which `lineage_rates()` requires before computing rate summaries from a mapped heterogeneous-rate history.

### Running `lineage_rates()`

With the weighting scheme established and the focal search loaded, the next step is to run the Berv et al. (2026) lineage-rate calculation. This call uses the parameter choices described above: `decay_base = 2`, `half_life = 5`, normalized weights, and log-scale rates. The progress bar is suppressed so the vignette output stays quiet.

Because each tip defines one extant lineage, the returned table has one row per species. The preview below sorts lineages by the inherited-history summary, `log_lineage_rate`, while keeping `log_tip_rate` beside it. That pairing is useful because both columns are on the log scale: the lineage summary reflects weighted ancestral history, whereas `log_tip_rate` records only the terminal fitted state.


In [ ]:
# Run the Berv et al. (2026) lineage-rate summary.
bodyplan_lineage_rates <- lineage_rates(
  bodyplan_search,
  progress = FALSE
)

# Show the fastest lineages first and keep the most useful columns.
lineage_preview <- bodyplan_lineage_rates[
  order(bodyplan_lineage_rates$log_lineage_rate, decreasing = TRUE),
  c("tip.label", "shift_count", "log_lineage_rate", "log_tip_rate")
]
head(lineage_preview)


To see how the temporal weighting changes this comparison, recompute the lineage summary with a much shorter half-life:


In [ ]:
bodyplan_lineage_rates_short_decay <- lineage_rates(
  bodyplan_search,
  half_life = 0.5,
  progress = FALSE
)


In [ ]:
lineage_tip_rate_theme <- ggplot2::theme_classic(base_size = 8.5) +
  ggplot2::theme(
    plot.margin = ggplot2::margin(2, 2, 2, 2),
    axis.title = ggplot2::element_text(color = "#263238", size = 11.05),
    axis.text = ggplot2::element_text(color = "#263238"),
    plot.title = ggplot2::element_text(color = "#263238", face = "bold", size = 9)
  )

lineage_tip_rate_panel <- function(rates, title, color) {
  plot_data <- data.frame(
    log_tip_rate = rates$log_tip_rate,
    log_lineage_rate = rates$log_lineage_rate
  )
  model <- lm(log_lineage_rate ~ log_tip_rate, data = plot_data)
  label <- sprintf(
    "y = %.2f + %.2fx\nR^2 = %.2f, r = %.2f",
    coef(model)[1],
    coef(model)[2],
    summary(model)$r.squared,
    cor(plot_data$log_tip_rate, plot_data$log_lineage_rate)
  )

  scatter <- ggplot2::ggplot(plot_data, ggplot2::aes(x = log_tip_rate, y = log_lineage_rate)) +
    ggplot2::geom_point(color = color, alpha = 0.22, size = 0.55) +
    ggplot2::geom_smooth(method = "lm", se = FALSE, color = "#9b2f2f", linewidth = 0.50) +
    ggplot2::annotate(
      "label",
      x = min(plot_data$log_tip_rate),
      y = max(plot_data$log_lineage_rate),
      label = label,
      hjust = 0,
      vjust = 1,
      size = 3.225,
      linewidth = 0,
      fill = "white",
      alpha = 0.82,
      color = "#263238"
    ) +
    ggplot2::coord_fixed() +
    ggplot2::labs(
      title = title,
      x = "Terminal fitted log tip rate",
      y = "Weighted log lineage rate"
    ) +
    lineage_tip_rate_theme

  x_density <- ggplot2::ggplot(plot_data, ggplot2::aes(x = log_tip_rate)) +
    ggplot2::geom_density(fill = color, color = color, alpha = 0.28, linewidth = 0.30) +
    ggplot2::theme_void() +
    ggplot2::theme(plot.margin = ggplot2::margin(0, 2, 0, 30))

  y_density <- ggplot2::ggplot(plot_data, ggplot2::aes(x = log_lineage_rate)) +
    ggplot2::geom_density(fill = color, color = color, alpha = 0.28, linewidth = 0.30) +
    ggplot2::coord_flip() +
    ggplot2::theme_void() +
    ggplot2::theme(plot.margin = ggplot2::margin(2, 0, 25, 0))

  patchwork::wrap_plots(
    x_density,
    patchwork::plot_spacer(),
    scatter,
    y_density,
    ncol = 2,
    widths = c(4, 0.72),
    heights = c(0.72, 4)
  )
}

lineage_tip_rate_default_panel <- lineage_tip_rate_panel(
  bodyplan_lineage_rates,
  "Default half-life: 5 Myr",
  "#386f8f"
)

lineage_tip_rate_extreme_panel <- lineage_tip_rate_panel(
  bodyplan_lineage_rates_short_decay,
  "Short half-life: 0.5 Myr",
  "#6f7f3b"
)

patchwork::wrap_plots(
  lineage_tip_rate_default_panel,
  lineage_tip_rate_extreme_panel,
  ncol = 2
)


You can merge the resulting species-level table with ecological, geographic, or climatic metadata when asking how lineage-level inferred rates relate to external predictors. Because these summaries are still estimated for species that share ancestry, phylogenetic methods such as PGLS may remain appropriate even though the estimator concentrates signal toward the present.

## Practical Takeaways

- `lineage_rates()` returns one inherited BMM rate-history summary per extant root-to-tip lineage.
- The Berv et al. (2026) lineage-rate statistic uses `decay_base = 2`, `half_life = 5`, normalized weights, and log-scale fitted rates.
- The age-decay kernel controls how strongly deeper ancestral history contributes to each lineage summary; shorter half-lives make the summary converge toward the terminal fitted rate.
- `log_tip_rate` is intentionally separate from `log_lineage_rate`: it records only the terminal fitted state on the same log scale.
- The lineage-rate table can be joined to species-level ecological, geographic, climatic, or trait metadata for downstream predictor analyses.
- Present-biased lineage-rate summaries do not remove phylogenetic non-independence; downstream models should still account for shared ancestry when the question calls for species-level inference.
- Continue to Part 3 for shift-transition tables, branch-rate annotations, fitted rate distributions, and waiting-time distributions.

## References

If you use these data or reproduce this workflow, the most relevant citations are:

- Berv, Jacob S., Charlotte M. Probst, Santiago Claramunt, J. Ryan Shipley, Matt Friedman, Stephen A. Smith, David F. Fouhey, and Brian C. Weeks. 2026. "Rates of passerine body plan evolution in time and space." *Nature Ecology & Evolution*. <https://doi.org/10.1038/s41559-026-03110-5>
- Berv, Jacob S., Charlotte M. Probst, Santiago Claramunt, J. Ryan Shipley, Matt Friedman, Stephen A. Smith, David F. Fouhey, and Brian C. Weeks. 2026. "Supplementary data archive for Rates of passerine body plan evolution in time and space" (v1.0.0) [Data set]. Zenodo. <https://doi.org/10.5281/zenodo.19198393>
- Claramunt, Santiago, Christopher Sheard, Joseph W. Brown, Guillermo Cortes-Ramirez, Joel Cracraft, Mason M. Su, Brian C. Weeks, and Joseph A. Tobias. 2025. "A new time tree of birds reveals the interplay between dispersal, geographic range size, and diversification." *Current Biology*. Advance online publication. <https://doi.org/10.1016/j.cub.2025.07.004>

## Software Used in This Vignette

- `bifrost` for weighted lineage-rate summaries from fitted BMM searches.
- `ape` and `phytools` underlie the mapped-tree calculations used by `lineage_rates()`.
- `ggplot2` and `patchwork` draw the age-decay and lineage-rate comparison figures.
